DAY 2: Data Pre-processing  

# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  


In [2]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.items import Item
from pricer.batch import Batch

In [2]:
LITE_MODE = True  

In [3]:
user_name = "ed-donner"
dataset =f"{user_name}/items_raw_lite" if LITE_MODE else f"{user_name}/items_raw_full"  

train,val,test = Item.from_hub(dataset)

items = train+ val + test

print(f"Loaded {len(items)} items")

Loaded 22000 items


In [4]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [5]:
items[2].id

In [6]:
for index,item in enumerate(items):
    item.id=index

In [7]:
items[2].id

2

In [8]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [9]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [10]:
messages = [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":items[0].full}]
response = completion(messages=messages,model="groq/openai/gpt-oss-20b",reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Title: Schlage F59 Interior Knob with Deadbolt (Oil Rubbed Bronze)  
Category: Hardware  
Brand: Schlage  
Description: A single-piece interior knob featuring an integrated deadbolt, finished in oil‑rubbed bronze for a classic look.  
Details: Designed for easy installation, it requires a F58 latch plate for completion and comes with a lifetime mechanical and finish warranty.

Input tokens: 446
Output tokens: 106
Cost: 0.010 cents


In [11]:


messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/codellama:7B", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Brand: Schlage
Title: Rewritten short precise title
Category: Electronics
Description: A two-piece handle set consisting of a deadbolt and an interior knob with oil rubbed bronze finish.
Details: Precision engineered, 100% solid design, easy to install, lifetime mechanical and finish warranty.

Input tokens: 478
Output tokens: 77
Cost: 0.000 cents


In [12]:
MODEL = "openai/gpt-oss-20b"

In [13]:
def make_Jsonl(item):
    body ={"model":MODEL,"messages":[{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":item.full}],"reasoning_effort":"low"}
    line ={"custom_id":str(item.id),"method":"POST","url":"v1/chat/completions","body":body}
    return json.dumps(line)


In [14]:
items[0].full

'Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dime

In [15]:
make_Jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two piece mo

In [16]:
def make_file(start,end,filename):
    with open(filename,"w") as f:
        for i in range(start,end):
            f.write(make_Jsonl(items[i]))
            f.write("\n")




In [18]:
make_file(0,1000,"data/items_0_100.jsonl")

In [19]:
import os
from groq import Groq

groq = Groq(api_key=os.environ["GROQ_API_KEY"])

In [23]:
with open("data/items_0_100.jsonl","rb") as f:
    response = groq.files.create(file=f,purpose="batch")
response

PermissionDeniedError: Error code: 403 - {'error': {'message': 'Not available for your plan', 'type': 'permissions_error', 'code': 'not_available_for_plan'}}

In [ ]:
file_id = response.id
file_id

In [ ]:
response = groq.batch.create(completion_window="24h",endpoint="v1/chat/completions",input_file_id=file_id)
response

In [ ]:
result = groq.batch.retrieve(response.id)
result

In [ ]:
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [ ]:
with open("jsonl/batch_results.jsonl","r") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["choices"][0]["message"]["content"]
        items[id].summary = summary

In [ ]:
print(items[0].full)


In [ ]:
print(items[0].summary)

let use exisiting modules generated by ed bhai!

In [3]:
Batch.create(items, LITE_MODE)

NameError: name 'items' is not defined

In [ ]:
Batch.run()

In [ ]:
Batch.fetch()

In [ ]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

## Push the final dataset to the hub

In [ ]:
username = "ed-donner"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)